# 🚀 RADAR DIGITAL - ENTRENAMIENTO DE MODELOS

## Notebook central para entrenar los 4 modelos del sistema

### Modelos disponibles:
- **Clasificador** (TabNet) - Clasifica nivel de adicción
- **Predictor** (XGBoost) - Predice riesgo compuesto
- **Explicador** (MLP) - Aproxima importancias SHAP
- **Generador** (MLP + Reglas) - Genera recomendaciones

### Configuración:
Cambia `MODELO_A_ENTRENAR` para elegir qué modelo entrenar.

Opciones: `"clasificador"`, `"predictor"`, `"explicador"`, `"generador"`, `"todos"`

In [1]:
# CONFIGURACIÓN - ELIGE QUÉ ENTRENAR

# Opciones: "clasificador" | "predictor" | "explicador" | "generador" | "todos"
MODELO_A_ENTRENAR = "todos" 

# 1. IMPORTAR DEPENDENCIAS

import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Agregar ruta raíz
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

from utils import set_seed, get_device

# Importar funciones main de cada modelo
from models.clasificador import main as train_clasificador
from models.predictor import main as train_predictor
from models.explicador import main as train_explicador
from models.generador import main as train_generador

print("✅ Dependencias importadas correctamente")

✅ Dependencias importadas correctamente


In [2]:
# 2. CONFIGURACIÓN INICIAL

RESULTADOS_DIR = ROOT_DIR / 'resultados_evaluaciones'
set_seed(42)
device = get_device()
print(f"📱 Dispositivo: {device}")
print(f"📂 Ruta raíz: {ROOT_DIR}")

✅ Semilla fijada a 42
✅ GPU disponible: NVIDIA GeForce RTX 3050 6GB Laptop GPU
   Memoria GPU: 6.22 GB
📱 Dispositivo: cuda
📂 Ruta raíz: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales


In [3]:
# 3. FUNCIÓN PARA ENTRENAR MODELOS

def entrenar_modelo(modelo):
    """
    Entrena un modelo específico y retorna sus métricas.
    """
    print("\n" + "=" * 60)
    print(f"🚀 ENTRENANDO: {modelo.upper()}")
    print("=" * 60)
    
    if modelo == "clasificador":
        clasificador, acc = train_clasificador()
        return {"accuracy": acc, "status": "✅ OK"}
    elif modelo == "predictor":
        predictor, r2 = train_predictor()
        return {"r2": r2, "status": "✅ OK"}
    elif modelo == "explicador":
        explicador, r2 = train_explicador()
        return {"r2": r2, "status": "✅ OK"}
    elif modelo == "generador":
        generador, r2 = train_generador()
        return {"r2": r2, "status": "✅ OK"}
    else:
        raise ValueError(f"Modelo desconocido: {modelo}")


def entrenar_todos():
    """
    Entrena todos los modelos secuencialmente.
    """
    modelos = ["clasificador", "predictor", "explicador", "generador"]
    resultados = {}
    
    for modelo in modelos:
        try:
            resultados[modelo] = entrenar_modelo(modelo)
        except Exception as e:
            resultados[modelo] = {"error": str(e), "status": "❌ ERROR"}
    
    return resultados

print("✅ Funciones de entrenamiento definidas")

✅ Funciones de entrenamiento definidas


In [4]:
# 4. EJECUTAR ENTRENAMIENTO

if MODELO_A_ENTRENAR == "todos":
    print("\n" + "=" * 60)
    print("🚀 ENTRENANDO TODOS LOS MODELOS")
    print("=" * 60)
    print("⏳ Esto puede tomar varios minutos...")
    resultados = entrenar_todos()
else:
    print("\n" + "=" * 60)
    print(f"🚀 ENTRENANDO: {MODELO_A_ENTRENAR.upper()}")
    print("=" * 60)
    resultado = entrenar_modelo(MODELO_A_ENTRENAR)
    resultados = {MODELO_A_ENTRENAR: resultado}


🚀 ENTRENANDO TODOS LOS MODELOS
⏳ Esto puede tomar varios minutos...

🚀 ENTRENANDO: CLASIFICADOR
✅ GPU disponible: NVIDIA GeForce RTX 3050 6GB Laptop GPU
   Memoria GPU: 6.22 GB
✅ Semilla fijada a 42
🚀 CLASIFICADOR TABNET - RADAR DIGITAL
📱 Dispositivo: cuda

🎯 Características de TabNet:
   ✅ Arquitectura basada en atención sparsemax
   ✅ Interpretabilidad mediante máscaras de features
   ✅ Balanceo de clases con sample weights
   ✅ Early stopping con paciencia
   ✅ Selección automática de características
   ✅ Métricas: Accuracy, Precision, Recall, F1

📂 CARGANDO DATOS
✅ Dataset cargado: 7500 registros, 25 columnas
✅ Features: 18 variables
✅ Target: addiction_level_encoded
✅ Clases: ['None', 'Mild', 'Moderate', 'Severe']

📊 PREPARANDO DATOS
✅ Entrenamiento: 4500 registros
✅ Validación: 1500 registros
✅ Prueba: 1500 registros

📊 Distribución de clases en entrenamiento:
   Clase 0: 491 (10.9%)
   Clase 1: 825 (18.3%)
   Clase 2: 1724 (38.3%)
   Clase 3: 1460 (32.4%)

🧠 ENTRENANDO CLASIFIC

In [5]:
# 5. RESUMEN FINAL

import json
import pandas as pd

print("\n" + "=" * 60)
print("📊 RESUMEN DE ENTRENAMIENTO")
print("=" * 60)

resumen = []
for modelo, resultado in resultados.items():
    status = resultado.get('status', '❌')
    if status == '✅ OK':
        metricas = {k: v for k, v in resultado.items() if k != 'status'}
        print(f"\n📋 {modelo.upper()}:")
        for metrica, valor in metricas.items():
            print(f"   {metrica}: {valor:.4f}")
        resumen.append({"modelo": modelo, **metricas, "status": status})
    else:
        print(f"\n📋 {modelo.upper()}: ❌ ERROR")
        print(f"   Error: {resultado.get('error', 'Desconocido')}")
        resumen.append({"modelo": modelo, "status": status, "error": resultado.get('error', '')})

# Guardar resultados
df_resultados = pd.DataFrame(resumen)
df_resultados.to_csv(RESULTADOS_DIR / 'resultados_entrenamiento.csv', index=False)

with open(RESULTADOS_DIR / 'resultados_entrenamiento.json', 'w') as f:
    json.dump(resultados, f, indent=4)

print("\n" + "=" * 60)
print("✅ Entrenamiento completado!")
print(f"📂 Resultados guardados en: {RESULTADOS_DIR}/resultados_entrenamiento.csv")
print("=" * 60)


📊 RESUMEN DE ENTRENAMIENTO

📋 CLASIFICADOR:
   accuracy: 0.5253

📋 PREDICTOR:
   r2: 0.9953

📋 EXPLICADOR:
   r2: 0.4389

📋 GENERADOR:
   r2: 0.9085

✅ Entrenamiento completado!
📂 Resultados guardados en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/resultados_evaluaciones/resultados_entrenamiento.csv
